<a href="https://colab.research.google.com/github/Bargav12/Bargav12/blob/main/Tushar_video_to_text_transcription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install ffmpeg-python

In [ ]:
! pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 68.6 MB/s eta 0:00:00


In [ ]:
! pip install langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 2.9 MB/s eta 0:00:00


In [ ]:
! pip install pydub

In [ ]:
import subprocess

In [ ]:
video_path = "ted_talk.mp4"

audio_path = "test_aud.wav"

command = f"ffmpeg -i {video_path} -ab 160k -ar 44100 -vn {audio_path}"
try:
    output = subprocess.check_output(command, shell=True, stderr=subprocess.STDOUT)
    print("Audio extracted successfully.")
except subprocess.CalledProcessError as e:
    print("FFmpeg error:", e.output.decode())

Audio extracted successfully.


In [ ]:
import speech_recognition as sr

recognizer = sr.Recognizer()

with sr.AudioFile(audio_path) as source:
    audio_data = recognizer.record(source)

try:
    text = recognizer.recognize_google(audio_data)
    print("Transcribed Text:\n", text)
except sr.UnknownValueError:
    print("Could not understand the audio.")
    text = ""
except sr.RequestError as e:
    print(f"Request Error: {e}")
    text = ""


Transcribed Text:
 so what's new Mark how is your new job going to be honest I can't complain I really love the company that I am working for my coworkers are all really friendly and helpful they really help me feel welcome it's a really energetic and fun atmosphere my boss is hilarious and he's really flexible really how so he allows me to come in when I want and make my own hours I can also leave early if I start early there is no real dress code either I can wear jeans and a t-shirt if I want I can even wear shorts in the summer why it sounds really cool I can't stand wearing a suit every day which do you prefer working late or finishing early I really enjoyed the morning I love getting up early and going for a run there's nothing like watching the sunrise well drinking my morning coffee really I am opposite I love sleeping in I am most Alert in the evenings I'm a real Night Owl you know what they say the early bird catches the worm you know you could be right maybe I will try to go

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from getpass import getpass
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API')

# You can use this line in Colab or local securely
# GROQ_API_KEY = getpass("Enter your GROQ API key: ")

llm = ChatGroq(
    temperature=0,
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

# Define a simple prompt template for summarization
prompt_summary = PromptTemplate.from_template(
    """
    ### TRANSCRIBED SPOKEN CONTENT:
    {input_text}

    ### TASK:
    Provide a clear, concise, and informative summary of the above content.
    - Focus on the main ideas, key points, and important details.
    - Remove filler words or irrelevant information typically found in spoken language.
    - Maintain the original intent and meaning.
    - Write in a formal and neutral tone, suitable for professional or academic use.

    ### OUTPUT:
    Summary:
    """
)

# Build LangChain summarization pipeline
chain_summary = prompt_summary | llm

# Run summarization
if text:
    result = chain_summary.invoke({'input_text': text})
    print("Summarized Text:\n", result.content)
else:
    print("No transcribed text to summarize.")


Summarized Text:
 Summary:
Mark is enjoying his new job, appreciating the friendly and helpful coworkers, and the energetic atmosphere of the company. His boss is flexible, allowing him to create his own schedule and dress casually. Mark prefers working in the mornings, enjoying activities such as running and watching the sunrise, while his counterpart is more productive in the evenings. The conversation touches on the benefits of being an early riser, with the phrase "the early bird catches the worm" being mentioned, prompting Mark to consider adjusting his sleep schedule. Overall, the discussion revolves around Mark's positive work experience and the differing preferences for daily routines.


In [ ]:
final_summary = result.content

In [ ]:
# Save as plain text files
with open("transcription.txt", "w", encoding="utf-8") as f:
    f.write(text)

with open("summary.txt", "w", encoding="utf-8") as f:
    f.write(final_summary)  # or result.content if single part


In [ ]:
! pip install fpdf


  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=34aab06eb36d160e02a421ec3a676c5686ca50cde8cc3b05e229aa1564b712a6
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [ ]:
from fpdf import FPDF

def save_as_pdf(title, content, filename):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    pdf.multi_cell(0, 10, txt=title, align='L')
    pdf.ln()

    pdf.set_font("Arial", size=11)
    pdf.multi_cell(0, 10, txt=content)

    pdf.output(filename)

# Save transcription and summary as PDFs
save_as_pdf("Video Transcription", text, "transcription.pdf")
save_as_pdf("Video Summary", final_summary, "summary.pdf")


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_insight = PromptTemplate.from_template("""
### TRANSCRIBED TEXT:
{text}

### TASK:
Analyze the content and return the following as a valid JSON:
{{
  "topic": "...",
  "sentiment": "...",  // Positive / Neutral / Negative
  "what_it_conveys": "...",
  "why_it_is_useful": "...",
  "use_cases": ["...", "...", "..."]
  "target audience"
}}

### JSON (NO explanation):
""")


In [ ]:
insight_chain = prompt_insight | llm
insight_response = insight_chain.invoke({'text': text})

import json

try:
    # Parse JSON string safely
    insights = json.loads(insight_response.content)
    print("Extracted Insights:\n", json.dumps(insights, indent=2))
except json.JSONDecodeError:
    print("Failed to parse JSON. Raw output:\n", insight_response.content)
    insights = {}


Failed to parse JSON. Raw output:
 ```json
{
  "topic": "Job satisfaction and work-life balance",
  "sentiment": "Positive",
  "what_it_conveys": "The importance of a flexible and enjoyable work environment in boosting job satisfaction",
  "why_it_is_useful": "It highlights the benefits of a relaxed work atmosphere, flexible hours, and a supportive team, which can be useful for employers and employees alike",
  "use_cases": [
    "Improving employee retention and recruitment",
    "Enhancing work-life balance and overall well-being",
    "Increasing productivity and job satisfaction"
  ],
  "target_audience": "Employers, employees, and HR professionals"
}
```


In [ ]:
# with open("video_insights.json", "w", encoding="utf-8") as f:
#     json.dump(insights, f, indent=2, ensure_ascii=False)
